# [센서 실습 9] 가스 센서 – MQ-2 (LPG·부탄·연기)

반도체식 가스 센서 **MQ-2**의 출력 전압을 ADC로 읽고, 데이터시트의 **특성 곡선**으로 가스 농도(ppm)를 계산하여 가스 누출을 감지합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
MQ-2 안에는 **산화주석(SnO₂)** 감지 소자와 이를 데우는 **히터**가 있습니다.
- 깨끗한 공기에서는 감지 소자의 저항(Rs)이 크고, 가연성 가스가 닿으면 **저항이 작아짐**
- 모듈은 부하저항(RL)과 Rs로 전압 분배 회로를 만들어, 가스가 많을수록 **출력 전압이 높아짐**
- 농도 계산 순서

$$V_{out} \;\rightarrow\; R_s = \frac{V_c - V_{out}}{V_{out}} \times R_L \;\rightarrow\; \frac{R_s}{R_0} \;\rightarrow\; \text{ppm} = 10^{\frac{\log_{10}(R_s/R_0) - b}{m}}$$

- $R_0$: 깨끗한 공기에서 보정한 기준 저항 (센서마다 다름 → **보정 필수**)
- $m, b$: 데이터시트 그래프(log-log)에서 읽은 직선의 기울기와 절편
- 히터가 데워져야 정확 → 처음 사용 시 **24~48시간 예열**, 매번 켤 때도 몇 분 예열 필요

### 부품 사양
| 항목 | 내용 |
|---|---|
| 감지 가스 | LPG, 부탄, 프로판, 메탄, 수소, 연기 |
| 감지 범위 | 300 ~ 10,000 ppm |
| 히터 전원 | 5V, 약 150mA |
| 출력 | 아날로그 0 ~ 5V (+ 디지털 D0) |
| 예열 | 최초 24시간 이상 권장 |

## 2. 라즈베리파이 연결 방법
**준비물**: MQ-2 모듈, MCP3008 ADC, 저항 10kΩ·20kΩ(전압 분배), 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MCP3008 VDD, VREF (16, 15번) | 3.3V (1번 핀) | ADC 전원과 기준 전압 |
| MCP3008 AGND, DGND (14, 9번) | GND (6번 핀) |  |
| MCP3008 CLK (13번) | GPIO11 / SCLK (23번 핀) | SPI 클록 |
| MCP3008 DOUT (12번) | GPIO9 / MISO (21번 핀) | SPI 데이터 (ADC → Pi) |
| MCP3008 DIN (11번) | GPIO10 / MOSI (19번 핀) | SPI 데이터 (Pi → ADC) |
| MCP3008 CS (10번) | GPIO8 / CE0 (24번 핀) | 칩 선택 |
| MQ-2 VCC / GND | 5V (2번 핀) / GND | 히터 때문에 5V 필요 |
| MQ-2 A0 → 10kΩ → MCP3008 CH0 | CH0 ↔ GND 사이 20kΩ | 5V 출력을 2/3로 낮춤 |

> ⚠️ MQ-2는 5V로 동작하므로 출력도 최대 5V입니다. **10kΩ/20kΩ 전압 분배**로 3.3V 이하로 낮춰 MCP3008에 넣고, 계산할 때 다시 1.5배 합니다.

**인터페이스 활성화**
- **SPI**: `sudo raspi-config` → Interface Options → **SPI** → Enable

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
ADC 값 → 실제 출력 전압 → 센서 저항 Rs → Rs/R0 → ppm 순서로 계산합니다. LPG 곡선 값을 사용했습니다.

In [ ]:
VC, RL, R0 = 5.0, 5.0, 10.0      # 센서 전원(V), 부하저항(kΩ), 깨끗한 공기에서 보정한 R0(kΩ)
DIVIDER = 1.5                     # 전압 분배로 2/3 줄였으므로 다시 1.5배
M_LPG, B_LPG = -0.47, 1.31        # 데이터시트 LPG 곡선: log10(Rs/R0) = m·log10(ppm) + b

def adc_to_ppm_mq2(adc_value, vref=3.3):
    v_out = max(adc_value * vref * DIVIDER, 1e-3)
    rs = (VC - v_out) / v_out * RL
    return 10 ** ((np.log10(rs / R0) - B_LPG) / M_LPG)

def calibrate_r0(adc_clean, vref=3.3, clean_air_ratio=9.8):
    """깨끗한 공기에서 읽은 ADC 값으로 R0 구하기 (데이터시트: 깨끗한 공기 Rs/R0 ≈ 9.8)"""
    v_out = adc_clean * vref * DIVIDER
    return (VC - v_out) / v_out * RL / clean_air_ratio

for a in (0.05, 0.15, 0.25, 0.40):
    print(f"ADC {a:.2f} → {adc_to_ppm_mq2(a):8.0f} ppm")

### 4-2. 실제 센서 클래스
ADC 값을 읽어 변환식으로 ppm을 구합니다. 처음 설치 때는 깨끗한 공기에서 `calibrate_r0()`로 R0를 구해 위의 `R0` 값을 바꿔야 합니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealMQ2:
    def __init__(self, channel=0):
        from gpiozero import MCP3008
        self.adc = MCP3008(channel=channel)

    def read(self):
        v = self.adc.value
        return {"adc": round(v, 4), "ppm": round(adc_to_ppm_mq2(v), 1)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 주방, 1시간 관찰 | 휴대용 버너 사용 |
| 평상시 | 약 10 ppm | 깨끗한 공기 (센서 최저 감지 수준) |
| 예열 | 처음 3분은 값이 높게 출렁임 | 히터가 데워지기 전의 실제 특성 |
| 가스 누출 | 25분부터 밸브가 열려 최대 약 3,000 ppm으로 상승 | 버너 밸브를 잠그지 않은 상황 |
| 환기 | 38분에 창문 열어 지수적으로 감소 | 환기 효과 |

가상 센서는 농도(ppm)를 **특성 곡선을 거꾸로 적용**해 ADC 값으로 바꾼 뒤, 실제와 같은 변환식으로 다시 ppm을 구합니다.

In [ ]:
class VirtualMQ2:
    def __init__(self, dt=30, seed=9):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    @staticmethod
    def ppm_to_adc(ppm, vref=3.3):
        """특성 곡선의 역계산: ppm → Rs → 출력 전압 → ADC 값"""
        rs = 10 ** (M_LPG * np.log10(ppm) + B_LPG) * R0
        v_out = VC * RL / (RL + rs)
        return v_out / DIVIDER / vref

    def read(self):
        m = self.t / 60
        ppm = 10.0
        if 25 <= m < 38:
            ppm += 3000 * (1 - np.exp(-(m - 25) / 4))            # 누출: 점점 증가
        elif m >= 38:
            peak = 3000 * (1 - np.exp(-13 / 4))
            ppm += peak * np.exp(-(m - 38) / 3)                  # 환기: 지수적 감소
        if m < 3:
            ppm *= self.rng.uniform(3, 15)                       # 예열 중 불안정
        ppm *= np.exp(self.rng.normal(0, 0.05))
        adc = round(float(self.ppm_to_adc(ppm)), 4)
        self.t += self.dt
        return {"adc": adc, "ppm": round(adc_to_ppm_mq2(adc), 1)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualMQ2(dt=30)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
30초 간격으로 1시간 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 120               # 측정 횟수
INTERVAL = 30               # 측정 간격(초)

sensor = RealMQ2() if USE_REAL_SENSOR else VirtualMQ2(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. 처음 **3분은 예열 구간**으로 판정에서 제외합니다.
2. **1,000 ppm 이상**이면 가스 누출 경보, 해제는 300 ppm 미만일 때 합니다.

> 가정용 가스 경보기는 폭발이 일어나는 농도(LPG 약 18,000 ppm)보다 **훨씬 낮은 농도**에서 울리도록 만듭니다. 여기서는 교육용으로 1,000 ppm을 기준으로 했습니다.

In [ ]:
df["min"] = df["time_s"] / 60
WARMUP, ALARM_ON, ALARM_OFF = 3, 1000, 300
alarm, st = False, []
for m, p in zip(df["min"], df["ppm"]):
    if m < WARMUP:
        st.append(0); continue
    if not alarm and p >= ALARM_ON: alarm = True
    elif alarm and p < ALARM_OFF: alarm = False
    st.append(int(alarm))
df["alarm"] = st
on = df[df["alarm"].diff() == 1]
off = df[df["alarm"].diff() == -1]
for _, r in on.iterrows(): print(f"🚨 {r['min']:.1f}분 가스 누출 경보 ({r['ppm']:.0f} ppm) → 밸브 차단·환기")
for _, r in off.iterrows(): print(f"✅ {r['min']:.1f}분 경보 해제 ({r['ppm']:.0f} ppm)")
print(f"최고 농도: {df.loc[df['min'] >= WARMUP, 'ppm'].max():.0f} ppm")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.semilogy(df["min"], df["ppm"], "k-", lw=1.5, label="LPG 농도")
ax.axhline(ALARM_ON, color="gray", ls="--", label="경보 1000 ppm")
ax.axhline(ALARM_OFF, color="gray", ls=":", label="해제 300 ppm")
ax.axvspan(0, WARMUP, color="lightgray", alpha=0.6); ax.text(0.3, 20, "예열", fontsize=9)
ax.set_title("MQ-2 가스 농도 (로그 눈금)"); ax.set_xlabel("시간 (분)"); ax.set_ylabel("ppm")
ax.grid(alpha=0.3, which="both"); ax.legend(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "가스센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "MQ-2 (LPG·부탄·연기)",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/gas", payload)
# client.disconnect()

## 직접 해 보기
- `R0`를 8.0, 12.0으로 바꾸면 같은 ADC 값의 ppm이 얼마나 달라지나요? (보정이 중요한 이유)
- 가스 누출 경보가 나면 GPIO 릴레이로 가스 밸브를 잠그는 코드를 생각해 보세요.
- 로그 눈금 대신 일반 눈금으로 그래프를 그려 차이를 비교해 보세요.